# Retrotraducción con más datos sintéticos (R6) — más paráfrasis por oración, dentro de cada fold

Repite la retrotraducción dentro de cada fold, pero con **varias paráfrasis por oración** en vez de una (`--multiplicador`). Cada
paráfrasis adicional se obtiene volviendo a muestrear el paso inglés → español de Helsinki-NLP con otra semilla y traduciéndola al shiwilu con
el NMT de F. Prado. Todo lo hace `validacion_cruzada_en_linea.py`; las paráfrasis repetidas se descartan.

**Estrategia (la misma que el entrenamiento): test / pool / core / dev.** Para cada fold: *test* = el fold (no se toca);
*pool* = los otros 4 folds; el pool se parte en *dev interno* (~1/6) y *core*. Hay dos etapas:
- **Etapa core (elegir `C`):** el aumento se genera/filtra solo con el core y se mide en el dev interno.
- **Etapa pool (modelo final):** el aumento se genera/filtra con todo el pool y se mide en el test.

Así, el dev interno nunca influye en el aumento con el que se elige `C`. En todas las etapas se descartan las filas sintéticas
idénticas a una oración del test.

**Cuánto aumentar.** Hoy Retrotraducción ya aporta ~85% del tamaño del pool real (~480 filas por fold); Mixup, 100%. Con `MULTIPLICADORES = [2]`
el aporte sube a ~170% (más sintético que real). **No te recomiendo subir más de 2**, y antes de insistir conviene aplicar filtros de calidad a las
paráfrasis: con 1 paráfrasis por oración ya hay ruido (paráfrasis que inventan contenido, duplicados) y Retrotraducción empeora a mBERT de forma
significativa; más volumen del mismo material tiende a amplificar ese ruido. Este cuaderno sirve para comprobarlo.

**Antes de correr:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU** (T4 sirve).
2. `git commit` + `git push` del repo con `validacion_cruzada_en_linea.py` actualizado (`--multiplicador`).
3. Checkpoint en Drive: `Mi unidad/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl/` (o entrénalo en el paso 3B).
4. Lo generado en la corrida anterior (`shiwilu_en_linea/retrotraduccion/fold<N>_pool.csv`) se **reutiliza**: solo se genera la paráfrasis extra.

**Tiempo:** ~15-30 min por fold y por paráfrasis extra (≈ 1.5-2.5 h para `[2]`), más ~15 min de evaluación. Cada archivo se guarda en Drive apenas
se produce; si Colab se corta, vuelve a correr el paso 4.

**Resultado:** un solo `.zip` con lo generado de todos los niveles y, por cada nivel, las predicciones, el resumen, las comparaciones pareadas, las curvas ROC
(CSV, grilla e imagen por experimento) y el volumen sintético usado.

## 0. Montar Google Drive (aquí queda guardado lo generado)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repo y el de F. Prado

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git clone https://github.com/fapi19/Tesis_Spa-Jeb.git oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
!git log --oneline -3

ruta = '/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py'
assert os.path.exists(ruta), ('No existe validacion_cruzada_en_linea.py: haz git commit + git push del repo reorganizado '
                              '(carpetas oe2_...) con los cambios nuevos y vuelve a correr esta celda.')
print('OK')

## 2. Entorno aislado de F. Prado (sus versiones fijadas; evita conflictos con los paquetes de Colab)

In [ ]:
%cd /content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
!pip install -q virtualenv
!rm -rf /content/nmt_venv
!virtualenv -q /content/nmt_venv
!/content/nmt_venv/bin/pip install -q -r requirements/nmt.txt
!/content/nmt_venv/bin/pip install -q sentence-transformers scikit-learn matplotlib pandas
!/content/nmt_venv/bin/python -c "import torch; print('GPU disponible:', torch.cuda.is_available())"
%cd /content/shiwilu-tesis

## 3A. Copiar el checkpoint desde Drive al disco local (recomendado)

In [ ]:
CHECKPOINT = "/content/checkpoint_retro/nllb_bidi_lora_v2_1b_loraplus_xl"

!mkdir -p /content/checkpoint_retro
!cp -r "/content/drive/MyDrive/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl" /content/checkpoint_retro/
for f in ["adapter_model.safetensors", "adapter_config.json", "tokenizer_config.json"]:
    assert os.path.exists(os.path.join(CHECKPOINT, f)), f"Falta {f} en {CHECKPOINT}."
tam = os.path.getsize(os.path.join(CHECKPOINT, "adapter_model.safetensors")) / 1e9
assert tam > 1.5, f"adapter_model.safetensors pesa solo {tam:.2f} GB (deberia ser ~1.97 GB): copia incompleta."
print(f"OK: checkpoint completo ({tam:.2f} GB) en {CHECKPOINT}")

## 3B. (OPCIONAL) Entrenar el checkpoint desde cero — SOLO si NO lo tienes en Drive

Está protegido: con `ENTRENAR = False` (por defecto) esta celda no hace nada, así que "Ejecutar todo" es seguro.

In [ ]:
ENTRENAR = False   # pon True SOLO si no tienes el checkpoint en Drive (y entonces omite el paso 3A)

if ENTRENAR:
    %cd /content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
    !/content/nmt_venv/bin/python -m scripts.nmt.30_train_lora --variant xl --rank 32 --alpha 64 --loraplus-lr-ratio 16 --output-dir models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl
    !mkdir -p /content/drive/MyDrive/shiwilu_checkpoint
    !cp -r models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl /content/drive/MyDrive/shiwilu_checkpoint/
    CHECKPOINT = "/content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb/models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl"
    %cd /content/shiwilu-tesis
else:
    print("3B omitido (ENTRENAR = False).")

## 4. Generar y evaluar, por nivel de volumen (todo en una corrida)

`MULTIPLICADORES` = paráfrasis por oración (1 ya está hecho). Para cada nivel y fold: reutiliza las paráfrasis ya guardadas, genera solo las que
faltan (y las guarda en Drive), elimina duplicados y copias del test, filtra por etapa, extrae embeddings y entrena. El paso de traducción
**no imprime progreso**: puede pasar mucho rato sin mostrar nada, y eso no significa que esté colgado.

In [ ]:
import os, subprocess

def correr(comando, cwd="/content/shiwilu-tesis"):
    """Corre un comando mostrando la salida en vivo; se detiene si falla."""
    proc = subprocess.Popen(
        comando, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"},
    )
    for linea in proc.stdout:
        print(linea, end="")
    proc.wait()
    assert proc.returncode == 0, f"Fallo (codigo {proc.returncode}): {' '.join(comando)}"

assert "CHECKPOINT" in globals(), "Falta correr el paso 3A (o 3B): define la variable CHECKPOINT."
PYTHON_NMT = "/content/nmt_venv/bin/python"

MULTIPLICADORES = [2]   # paráfrasis por oración; 1 = la corrida anterior (ya hecha)
PRUEBA = False          # True = prueba rapida: solo el fold 0, con cache y etiquetas aparte (no contamina la corrida completa)
CACHE = "/content/drive/MyDrive/shiwilu_en_linea" + ("_prueba" if PRUEBA else "")
SUFIJO = "_prueba" if PRUEBA else ""
NIVELES = MULTIPLICADORES
TAGS = [f"en_linea_retrotraduccion_colab_x{m}{SUFIJO}" for m in MULTIPLICADORES]

for m in MULTIPLICADORES:
    print(f"\n=============== Retrotraduccion, {m} paráfrasis por oración ===============")
    correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py",
            "--tecnica", "retrotraduccion", "--checkpoint", CHECKPOINT, "--multiplicador", str(m),
            "--cache", CACHE, "--etiqueta", f"colab_x{m}{SUFIJO}"] + (["--folds", "0"] if PRUEBA else []))

## 5. Tablas y curvas ROC (las mismas que el repo), por nivel

El CSV de predicciones de cada nivel trae **los 12 experimentos** (la retrotraducción de ese nivel + las otras tres configuraciones vigentes). Por cada
nivel se generan `comparacion_pareada_sin_puntuacion_<TAG>.csv`, `curvas_roc_<TAG>.csv`, `curvas_roc_<TAG>.png` (grilla 3 × 4) y la carpeta
`curvas_roc_<TAG>/` con una imagen por experimento.

In [ ]:
if not PRUEBA:
    import pandas as pd
    from IPython.display import display, Image
    RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"

    for TAG in TAGS:
        ENTRADA = f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv"
        print("=" * 20, TAG, "=" * 20)
        correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/comparacion_pareada.py", "--entrada", ENTRADA, "--etiqueta", TAG])
        correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/curvas_roc.py", "--entrada", ENTRADA, "--etiqueta", TAG])
        display(Image(f"{RES}/curvas_roc_{TAG}.png"))
        auc_macro = pd.read_csv(f"{RES}/curvas_roc_{TAG}.csv").query("categoria == 'macro'")
        display(auc_macro.pivot(index="modelo", columns="tecnica", values="auc").round(4))

if PRUEBA:
    print('Modo prueba: solo se evaluo el fold 0; las tablas y curvas ROC necesitan los 5 folds.')

## 6. Volumen usado y comparación frente a `sin_aumento` y a la retrotraducción vigente

In [ ]:
if not PRUEBA:
    import sys
    import numpy as np
    sys.path.insert(0, "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion")
    from comparacion_pareada import diferencia_pareada

    P_vig = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion.csv")
    vigente = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion.csv")
    def f1(tabla, tecnica):
        return tabla[tabla.tecnica == tecnica].set_index("modelo").f1_macro_agrupado

    columnas = {"sin aumento": f1(vigente, "sin_aumento"), "retro vigente (20, filtro original)": f1(vigente, "retrotraduccion")}
    pcts = {}
    for nivel, TAG in zip(NIVELES, TAGS):
        r = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion_{TAG}.csv")
        columnas[f"retro x {nivel}"] = f1(r, "retrotraduccion")
        v = pd.read_csv(f"{RES}/volumen_sintetico_sin_puntuacion_{TAG}.csv")
        pcts[nivel] = round(v[v.etapa == "pool"].pct_de_los_reales.mean(), 1)
    display(pd.DataFrame(columnas).round(4))
    print("Volumen sintetico usado en la etapa pool, como % de las oraciones reales del pool:", pcts)

    rng = np.random.default_rng(42)
    filas = []
    for nivel, TAG in zip(NIVELES, TAGS):
        P_nuevo = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv")
        P = pd.concat([P_nuevo[P_nuevo.tecnica == "retrotraduccion"].assign(tecnica="nuevo"),
                       P_vig[P_vig.tecnica == "sin_aumento"],
                       P_vig[P_vig.tecnica == "retrotraduccion"].assign(tecnica="vigente")])
        for modelo in ["labse", "mbert", "xlmr"]:
            for a, b in [("nuevo", "sin_aumento"), ("nuevo", "vigente")]:
                d, lo, hi = diferencia_pareada(P, (modelo, a), (modelo, b), rng)
                filas.append({"nivel": nivel, "modelo": modelo, "comparacion": f"{a} - {b}", "diferencia_f1": round(d, 4),
                              "ic95_bajo": round(lo, 4), "ic95_alto": round(hi, 4), "distinguible_de_cero": not (lo <= 0 <= hi)})
    display(pd.DataFrame(filas))

## 7. Descargar el resultado (un solo `.zip`)

Contiene `en_linea/retrotraduccion/` (lo generado, de todos los niveles) y `resultados/` (por nivel: predicciones y resumen con los 12 experimentos, comparaciones
pareadas, curvas ROC en CSV, grilla e imágenes por experimento, y el volumen sintético). En tu computadora: lo generado va en
`oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea/` y `resultados/` en `oe2_aumento_de_datos/evaluacion/resultados/`. Lo generado también queda en Drive.

In [ ]:
import glob, shutil
from google.colab import files

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
carpeta = "/content/salida_retro_en_linea"
shutil.rmtree(carpeta, ignore_errors=True)
os.makedirs(carpeta + "/resultados"); os.makedirs(carpeta + "/en_linea")
shutil.copytree(f"{CACHE}/retrotraduccion", carpeta + "/en_linea/retrotraduccion")      # lo generado, todos los niveles
for ruta_ in glob.glob(f"{RES}/*en_linea_retrotraduccion_colab_x*"):                             # CSV, PNG y carpetas de curvas ROC de todos los niveles
    destino = carpeta + "/resultados/" + os.path.basename(ruta_)
    shutil.copytree(ruta_, destino) if os.path.isdir(ruta_) else shutil.copy(ruta_, destino)
zip_path = shutil.make_archive("/content/salida_retro_en_linea", "zip", carpeta)
print(sorted(os.listdir(carpeta + "/resultados")))
files.download(zip_path)